In [1]:
!pip install papermill
import papermill as pm
from pathlib import Path
import pandas as pd
import time
from concurrent.futures import ProcessPoolExecutor, as_completed
import os
import uuid
import tempfile
import pathlib
from preprocess_data import CVSplitConfig, TimeSeriesCVSplitter

In [2]:
def infer_years_from_csv(csv_path: Path, date_col: str = "datetime") -> float:
    df = pd.read_csv(csv_path, usecols=[date_col])
    dt = pd.to_datetime(df[date_col]).sort_values()

    start = dt.iloc[0]
    end = dt.iloc[-1]

    months = (end.year - start.year) * 12 + (end.month - start.month) + 1
    years = months / 12.0

    if abs(years - round(years)) < 1e-9:
        years = float(int(round(years)))

    return years

In [3]:
def run_all_splits_with_papermill(
    notebook_in: str = "Notebook_Implement.ipynb",
    splits_dir: str = "CV_Data_Splits",
    notebook_out_dir: str = "output_notebooks",
    date_col: str = "datetime",
    subset_path_param: str = "subset_path",
    run_name_param: str = "run_name",
    forecast_param: str = "forecast_horizon",
    extra_params: dict | None = None,
):
    splits_dir = Path(splits_dir)
    notebook_out_dir = Path(notebook_out_dir)
    notebook_out_dir.mkdir(parents=True, exist_ok=True)
    train_files = sorted(splits_dir.glob("*_train.csv"))
    if not train_files:
        raise FileNotFoundError("No *_train.csv files found.")

    # -----------------------------
    # PRECOMPUTE ALL METADATA
    # -----------------------------
    summary = []
    for train_path in train_files:
        label = train_path.stem.replace("_train", "")
        test_path = train_path.with_name(train_path.name.replace("_train.csv", "_test.csv"))
        if not test_path.exists():
            raise FileNotFoundError(f"Missing test set for {label}: {test_path.name}")
        train_years = infer_years_from_csv(train_path, date_col=date_col)
        test_years  = infer_years_from_csv(test_path,  date_col=date_col)
        forecast_horizon = int(round(12 * test_years))
        summary.append(
            dict(
                label=label,
                train_path=train_path,
                train_years=train_years,
                test_years=test_years,
                forecast_horizon=forecast_horizon,
            )
        )

    # -----------------------------
    # PRINT SANITY-CHECK TABLE
    # -----------------------------
    print("\n=== CV SPLIT SUMMARY ===")
    print("label       | train_yrs | test_yrs | horizon (mo) | train_file")
    print("-" * 72)
    for s in summary:
        print(
            f"{s['label']:11s} | "
            f"{s['train_years']:9.1f} | "
            f"{s['test_years']:8.1f} | "
            f"{s['forecast_horizon']:12d} | "
            f"{s['train_path'].name}"
        )
    print("-" * 72 + "\n")

    # -----------------------------
    # RUN PAPERMILL
    # -----------------------------
    failures = []

    for s in summary:
        out_path = notebook_out_dir / f"{s['label']}.ipynb"
        params = {
            subset_path_param: str(s["train_path"]),
            forecast_param: s["forecast_horizon"],
            run_name_param: s["label"],
        }
        if extra_params:
            params.update(extra_params)

        print(
            f"Running {s['label']} "
            f"(train={s['train_years']} yrs, "
            f"test={s['test_years']} yrs, "
            f"horizon={s['forecast_horizon']} mo)"
        )

        try:
            pm.execute_notebook(
                notebook_in,
                str(out_path),
                parameters=params,
                report_mode=False,
            )
            print(f"  ✓ {s['label']} completed")
        except Exception as e:
            msg = str(e).splitlines()[0]
            print(f"  ✗ {s['label']} FAILED: {msg}")
            print(f"    → see {out_path} for full traceback")
            failures.append({"label": s["label"], "error": msg, "notebook": str(out_path)})

    # -----------------------------
    # FINAL SUMMARY
    # -----------------------------
    print("\n=== RUN COMPLETE ===")
    total = len(summary)
    n_fail = len(failures)
    print(f"  {total - n_fail}/{total} splits succeeded")
    if failures:
        print(f"\n  Failed splits ({n_fail}):")
        for f in failures:
            print(f"    • {f['label']:11s}  →  {f['error']}")
            print(f"                  notebook: {f['notebook']}")
    print()

    return failures

In [4]:
build_baseline = False
build_new_splits = False

In [5]:
if build_baseline:
    cfg = CVSplitConfig(
        csv_path="daily_flows_wide_by_gauge_name.csv", #"Discharge_Preprocess.csv"
        out_dir="CV_Data_Splits_All",
        date_col="datetime",
    )
    
    TimeSeriesCVSplitter(cfg).load().write_all()

if build_new_splits:
    cfg = CVSplitConfig(
        csv_path="Discharge_Preprocess.csv",
        out_dir="CV_Data_Splits_25_5",
        date_col="datetime",

    # 1) Consecutive
    consecutive_train_years=30.0,
    consecutive_test_years=5.0,

    # 2) Building blocks
    building_block_specs=[
        (40.0, 10.0),
        (55.0, 10.0),
        (70.0, 10.0),
    ],

    # 3) Standard
    standard_spec=(75.0, 15.0),
    )
    
    TimeSeriesCVSplitter(cfg).load().write_all()

In [6]:
# --- Example call ---
run_all_splits_with_papermill(
    notebook_in="Notebook_Implement_joint_thin.ipynb",
    splits_dir="CV_Data_Splits_All",
    notebook_out_dir="output_notebooks",
    date_col="datetime",
    subset_path_param="discharge_path",     # <-- must match the parameter name in your notebook
    extra_params={
        "experiment":"default",
        "parent_dir": "NEW_CV_ALL_param_spline",          # keep your existing export root
        "export_results": True,
        "test_frac": 0.0,
    },
)

Unable to parse line 3 'forecast_horizon = 60            # months to forecast (default 60 = 5 years)'.
Unable to parse line 43 'joint_K          = None           # None = auto sqrt(n_windows)'.
Passed unknown parameter: forecast_horizon



=== CV SPLIT SUMMARY ===
label       | train_yrs | test_yrs | horizon (mo) | train_file
------------------------------------------------------------------------
block1      |      50.0 |      4.0 |           48 | block1_train.csv
block2      |      60.0 |      5.0 |           60 | block2_train.csv
block3      |      70.0 |      6.0 |           72 | block3_train.csv
block4      |      80.0 |      7.0 |           84 | block4_train.csv
split1      |      25.0 |      2.0 |           24 | split1_train.csv
split2      |      25.0 |      2.0 |           24 | split2_train.csv
split3      |      25.0 |      2.0 |           24 | split3_train.csv
std         |      81.0 |      8.0 |           96 | std_train.csv
------------------------------------------------------------------------

Running block1 (train=50.0 yrs, test=4.0 yrs, horizon=48 mo)


Executing:   0%|          | 0/46 [00:00<?, ?cell/s]

Unable to parse line 3 'forecast_horizon = 60            # months to forecast (default 60 = 5 years)'.
Unable to parse line 43 'joint_K          = None           # None = auto sqrt(n_windows)'.
Passed unknown parameter: forecast_horizon


  ✓ block1 completed
Running block2 (train=60.0 yrs, test=5.0 yrs, horizon=60 mo)


...

Unable to parse line 3 'forecast_horizon = 60            # months to forecast (default 60 = 5 years)'.
Unable to parse line 43 'joint_K          = None           # None = auto sqrt(n_windows)'.
Passed unknown parameter: forecast_horizon


  ✓ split3 completed
Running std (train=81.0 yrs, test=8.0 yrs, horizon=96 mo)


Executing:   0%|          | 0/46 [00:00<?, ?cell/s]

  ✓ std completed

=== RUN COMPLETE ===
  8/8 splits succeeded



[]